In [2]:
import glob
import pandas as pd

In [3]:
file_paths = glob.glob("data/old/*.csv")

df_list = [
    pd.read_csv(f, usecols=["RollNo", "Name"]) for f in sorted(file_paths)
]

In [4]:
combined_df = (
    pd.concat(df_list, ignore_index=True)
    .assign(
        RollNo=lambda df: df["RollNo"].astype(str).str.strip(),
        Name=lambda df: df["Name"].astype(str).str.strip(),
    )
    .drop_duplicates(subset=["RollNo"])
    .sort_values(
        by="RollNo",
        key=lambda s: s.str[-5:].astype(int),
        ascending=True,
    )
    .reset_index(drop=True)
)

In [5]:
combined_df.to_csv("data/people.csv", index=False)

In [9]:
traffic_df = pd.read_csv("data/scrapped_data.csv")

extracted_roll = (
    "IMS" + traffic_df["Password"].astype(str).str.strip().str[:-2].str[-5:]
)

roll_to_name = dict(zip(combined_df["RollNo"], combined_df["Name"].str.title()))

traffic_df["Name"] = (
    extracted_roll.map(roll_to_name).fillna(traffic_df["Name"].str.title())
)

traffic_df.to_csv("data/scrapped_updated.csv", index=False)